In [15]:
import os
from dotenv import load_dotenv
from pathlib import Path
from langchain.chat_models import init_chat_model
import pandas as pd

## Config

In [63]:
# Nothing works without an API key
load_dotenv()

if not os.environ["OPENAI_API_KEY"]:
    print("Please add your OpenAI API key to .env to proceed.")

In [67]:
# Setting constants for later use
MODEL_ID = "openai:gpt-5.4-nano"
DATA_PATH = "data/raw"
TRAIN_FILE = "train.csv"
TEST_FILE = "test.csv"
VAL_FILE = "val.csv"
TOKEN_FACTOR = 1.3
TOKEN_COST_INPUT = 0.2 / 10 ** 6
TOKEN_COST_OUTPUT = 1.25 / 10 ** 6
PROMPT_TEMPLATE = "What is the sentiment? (Positive or Negative):"

In [ ]:
# Initialize our chosen model
model = init_chat_model(MODEL_ID)

## Loading the data

In [43]:
test_filepath = Path(DATA_PATH) / TEST_FILE
test_df = pd.read_csv(test_filepath)

In [44]:
test_df.head()

,Text,Label
0,I am always looking for a quick and easy but n...,1
1,"We had been giving our dog the Liver Biscotti,...",1
2,It has a very foul aftertaste. Taste at first ...,0
3,I love SoBe Fruit Punch. My neighborhood Tops ...,1
4,"When I lived in TN, I was able to find this pu...",1


In [52]:
test_df["text_length"] = test_df["Text"].str.len()

In [53]:
test_df.describe()

,Label,text_length
count,5000.000000,5000.000000
mean,0.842600,419.860600
std,0.364214,411.017433
min,0.000000,32.000000
25%,1.000000,175.000000
50%,1.000000,298.000000
75%,1.000000,513.000000
max,1.000000,5790.000000


In [68]:
# How many tokens are we about to send through this thing?
num_samples = len(test_df)
prompt_len = len(PROMPT_TEMPLATE)
mean_num_words = float(test_df.describe().loc["mean"]["text_length"])
estimated_num_tokens = num_samples * (mean_num_words + prompt_len) * TOKEN_FACTOR
print(f"Estimated number of tokens for test set: {estimated_num_tokens:.3f}")

Estimated number of tokens for test set: 3028093.900


In [71]:
# Let's think about the cost before we do that...
estimated_input_cost = estimated_num_tokens * TOKEN_COST_INPUT
print(f"Estimated cost of input tokens: {estimated_input_cost}")

estimated_output_cost = num_samples * TOKEN_COST_OUTPUT
print(f"Estimated cost of output tokens: {estimated_output_cost}")

estimated_total_cost = estimated_input_cost + estimated_output_cost
print(f"Estimated total cost of experiment: {estimated_total_cost}")

Estimated cost of input tokens: 0.60561878
Estimated cost of output tokens: 0.00625
Estimated total cost of experiment: 0.61186878


## Setting up the model

In [ ]:
# Let's test the model to make sure everything is working
review = test_df["Text"][0]

try:
    response = model.invoke("Hello")
    print(response)
except e:
    print(f"Error: {e}")

In [50]:
response.content

'Hello! How can I help you today?'

## Generating responses

## Evaluating the model responses

## References

- [LangChain init_chat_model](https://reference.langchain.com/python/langchain/chat_models/base/init_chat_model)
- [OpenAI API Pricing](https://openai.com/api/pricing/)